# SFT Practice: News Summarization

**Role:** end-to-end training notebook  
**Fine-tuning:** yes  
**Operation name:** `news-summarization-sft`

This notebook preserves the source demo's technical depth while using one restart-safe journey:
readiness, exact dataset validation, base baseline, upload/reuse, job create/reuse, monitoring,
deployment/invocation, evaluation, decision, and cleanup.

> **Cost warning:** inference and fine-tuning incur Azure charges. Job creation is blocked unless
> `FOUNDRY_RUN_PAID_JOBS` is explicitly set to `true`, and live inference is blocked unless
> `FOUNDRY_RUN_LIVE_EVALUATION` is `true`. Reuse a succeeded job whenever possible.


## Representative completed-run evidence

The chart and metrics below are sanitized, checked-in evidence from an existing run. They are
context for what a completed workflow can look like, not a promise for this run.

![Representative run](../assets/charts/representative-run.svg)

See [`representative-run.json`](../assets/metrics/representative-run.json) for scope and limitations.

*Actual results may vary by model version, data, configuration, region availability, and service conditions.*


## 1. Configuration and planned shared helpers

The optional `shared.*` imports match the planned cookbook helper surface. This notebook retains
visible local validation and orchestration code so readers can understand every operation today.


In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path
from urllib.parse import urlparse

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

try:
    from cookbooks.shared.config import load_foundry_config as shared_load_foundry_config
    from cookbooks.shared.dataset_validation import validate_jsonl as shared_validate_jsonl
    from cookbooks.shared.foundry_operations import (
        create_or_reuse_fine_tuning_job as shared_create_or_reuse_fine_tuning_job,
        monitor_fine_tuning_job as shared_monitor_fine_tuning_job,
        upload_or_reuse_file as shared_upload_or_reuse_file,
    )
except ModuleNotFoundError as exc:
    if exc.name and not (
        exc.name == "cookbooks" or exc.name.startswith("cookbooks.shared")
    ):
        raise
    shared_load_foundry_config = None
    shared_validate_jsonl = None
    shared_create_or_reuse_fine_tuning_job = None
    shared_monitor_fine_tuning_job = None
    shared_upload_or_reuse_file = None

load_dotenv()

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    DEMO_DIR = NOTEBOOK_DIR.parent
else:
    DEMO_DIR = NOTEBOOK_DIR
DATA_DIR = DEMO_DIR / "data"
OUTPUT_DIR = DEMO_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPERATION_NAME = "news-summarization-sft"
TRAINING_PATH = DATA_DIR / "news-summarization-train.jsonl"
VALIDATION_PATH = DATA_DIR / "news-summarization-validation.jsonl"
EXPECTED_TRAINING_SHA256 = "cf7e7cc1ed8b02f9959c269113bf396546c1e959599376c17095cee7a14ef7fd"
EXPECTED_VALIDATION_SHA256 = "25901f4b80de3f6c052f026b5184fca4057dfc6be4ca2be61e2c41f18efe92b7"
EXPECTED_TRAINING_ROWS = 1992
EXPECTED_VALIDATION_ROWS = 229
EVALUATION_SAMPLE_COUNT = 10

def required(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

def optional(name):
    value = os.getenv(name, "").strip()
    return value or None

def as_bool(name):
    value = os.getenv(name, "").strip().lower()
    if not value:
        return False
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be true, false, or blank")
    return value == "true"

config = {
    "project_endpoint": required("FOUNDRY_PROJECT_ENDPOINT"),
    "base_deployment_name": required("FOUNDRY_BASE_DEPLOYMENT"),
    "sft_base_model": required("FOUNDRY_BASE_MODEL"),
    "judge_deployment_name": optional("FOUNDRY_JUDGE_DEPLOYMENT"),
    "existing_training_file_id": optional("FOUNDRY_EXISTING_TRAINING_FILE_ID"),
    "existing_validation_file_id": optional("FOUNDRY_EXISTING_VALIDATION_FILE_ID"),
    "existing_job_id": optional("FOUNDRY_EXISTING_FINE_TUNE_JOB_ID"),
    "fine_tuned_deployment_name": optional("FOUNDRY_FINE_TUNED_DEPLOYMENT"),
    "allow_paid_job_creation": as_bool("FOUNDRY_RUN_PAID_JOBS"),
    "run_live_evaluation": as_bool("FOUNDRY_RUN_LIVE_EVALUATION"),
    "create_deployment": as_bool("FOUNDRY_CREATE_DEPLOYMENT"),
    "delete_created_deployment": as_bool("FOUNDRY_DELETE_CREATED_DEPLOYMENT"),
    "subscription_id": optional("FOUNDRY_SUBSCRIPTION_ID"),
    "resource_group": optional("FOUNDRY_RESOURCE_GROUP"),
    "deployment_sku": optional("FOUNDRY_DEPLOYMENT_SKU"),
    "deployment_capacity": optional("FOUNDRY_DEPLOYMENT_CAPACITY"),
}

if "/api/projects/" not in config["project_endpoint"]:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT must be a Microsoft Foundry project endpoint")
if bool(config["existing_training_file_id"]) != bool(config["existing_validation_file_id"]):
    raise ValueError("Set both existing file IDs, or leave both blank")

print(f"Configured operation: {OPERATION_NAME}")
print(f"Planned shared helpers available: {shared_load_foundry_config is not None}")


## 2. Authenticate and validate project prerequisites

`DefaultAzureCredential` uses the signed-in developer or managed identity. No API keys are read or
printed. This preflight verifies that the configured base and judge deployments are visible.
Fine-tuning model/region/quota support still must be confirmed in the current Foundry documentation
and project quota view because availability changes over time.


In [ ]:
credential = DefaultAzureCredential(process_timeout=120)
project_client = AIProjectClient(
    endpoint=config["project_endpoint"],
    credential=credential,
)
openai_client = project_client.get_openai_client()

deployment_names = {item.name for item in project_client.deployments.list()}
required_deployments = {config["base_deployment_name"]}
if config["judge_deployment_name"]:
    required_deployments.add(config["judge_deployment_name"])
missing_deployments = sorted(required_deployments - deployment_names)
if missing_deployments:
    raise RuntimeError(f"Required project deployments are missing: {missing_deployments}")

print("Authentication succeeded and required inference deployments are visible.")
print("Before creating a job, confirm fine-tuning access, regional model support, and training quota.")


## 3. Validate the exact preserved training and validation files

The copied files are byte-for-byte identical to the source demo. Validation checks SHA-256, row
count, JSON parsing, exact `system -> user -> assistant` role order, non-empty content, and duplicate
rows. The exact validated paths below are the only paths uploaded later.


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_chat_jsonl(path, expected_hash, expected_rows):
    actual_hash = file_sha256(path)
    if actual_hash != expected_hash:
        raise ValueError(f"Hash mismatch for {path.name}: {actual_hash}")

    rows = []
    row_hashes = set()
    duplicates = 0
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            messages = row.get("messages")
            if not isinstance(messages, list) or len(messages) != 3:
                raise ValueError(f"{path.name}:{line_number} must contain exactly three messages")
            roles = [message.get("role") for message in messages]
            if roles != ["system", "user", "assistant"]:
                raise ValueError(f"{path.name}:{line_number} has invalid role order: {roles}")
            if any(not isinstance(message.get("content"), str) or not message["content"].strip() for message in messages):
                raise ValueError(f"{path.name}:{line_number} has empty message content")
            canonical = json.dumps(row, sort_keys=True, ensure_ascii=False).encode("utf-8")
            row_hash = hashlib.sha256(canonical).hexdigest()
            duplicates += row_hash in row_hashes
            row_hashes.add(row_hash)
            rows.append(row)

    if len(rows) != expected_rows:
        raise ValueError(f"{path.name} has {len(rows)} rows; expected {expected_rows}")
    return {
        "path": str(path),
        "sha256": actual_hash,
        "rows": len(rows),
        "duplicate_rows": duplicates,
        "records": rows,
    }

training_report = validate_chat_jsonl(
    TRAINING_PATH,
    EXPECTED_TRAINING_SHA256,
    EXPECTED_TRAINING_ROWS,
)
validation_report = validate_chat_jsonl(
    VALIDATION_PATH,
    EXPECTED_VALIDATION_SHA256,
    EXPECTED_VALIDATION_ROWS,
)
training_rows = training_report.pop("records")
validation_rows = validation_report.pop("records")
dataset_report = {
    "training": training_report,
    "validation": validation_report,
    "hashes": {
        "training": training_report["sha256"],
        "validation": validation_report["sha256"],
    },
}
print(json.dumps(dataset_report, indent=2))


## 4. Establish the base-model baseline

The evaluation sample is the first fixed set of validation rows, preserving membership and making
base/fine-tuned comparisons repeatable. Because the source demo has no separate final holdout, this
is a validation-set comparison, not an independent estimate of generalization.


In [ ]:
import re

def normalized_tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def token_f1(candidate, reference):
    candidate_tokens = normalized_tokens(candidate)
    reference_tokens = normalized_tokens(reference)
    if not candidate_tokens or not reference_tokens:
        return 0.0
    candidate_counts = {}
    reference_counts = {}
    for token in candidate_tokens:
        candidate_counts[token] = candidate_counts.get(token, 0) + 1
    for token in reference_tokens:
        reference_counts[token] = reference_counts.get(token, 0) + 1
    overlap = sum(min(count, reference_counts.get(token, 0)) for token, count in candidate_counts.items())
    precision = overlap / len(candidate_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def lcs_length(left, right):
    previous = [0] * (len(right) + 1)
    for left_token in left:
        current = [0]
        for index, right_token in enumerate(right, start=1):
            if left_token == right_token:
                current.append(previous[index - 1] + 1)
            else:
                current.append(max(previous[index], current[-1]))
        previous = current
    return previous[-1]

def rouge_l_f1(candidate, reference):
    candidate_tokens = normalized_tokens(candidate)
    reference_tokens = normalized_tokens(reference)
    if not candidate_tokens or not reference_tokens:
        return 0.0
    overlap = lcs_length(candidate_tokens, reference_tokens)
    precision = overlap / len(candidate_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def domain_coverage(candidate, reference):
    return token_f1(candidate, reference)

def summarize_with(deployment_name, example):
    system_message, user_message = example["messages"][:2]
    response = openai_client.chat.completions.create(
        model=deployment_name,
        messages=[system_message, user_message],
        temperature=0,
        max_completion_tokens=600,
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError(f"{deployment_name} returned an empty summary")
    return content

def evaluate_deployment(deployment_name, examples, label):
    rows = []
    for index, example in enumerate(examples, start=1):
        reference = example["messages"][2]["content"]
        candidate = summarize_with(deployment_name, example)
        row = {
            "index": index,
            "token_f1": token_f1(candidate, reference),
            "rouge_l_f1": rouge_l_f1(candidate, reference),
            "domain_coverage": domain_coverage(candidate, reference),
            "candidate_words": len(normalized_tokens(candidate)),
            "reference_words": len(normalized_tokens(reference)),
        }
        rows.append(row)
        print(
            f"{label} example {index}: ROUGE-L={row['rouge_l_f1']:.3f}, "
            f"coverage={row['domain_coverage']:.3f}"
        )
    aggregate = {
        metric: sum(row[metric] for row in rows) / len(rows)
        for metric in ("token_f1", "rouge_l_f1", "domain_coverage")
    }
    return {"label": label, "aggregate": aggregate, "rows": rows}


In [ ]:
evaluation_examples = validation_rows[:EVALUATION_SAMPLE_COUNT]
baseline_results = None
if config["run_live_evaluation"]:
    baseline_results = evaluate_deployment(
        config["base_deployment_name"],
        evaluation_examples,
        "base",
    )
    print(json.dumps(baseline_results["aggregate"], indent=2))
else:
    print("Live baseline skipped. Review ../assets/metrics/representative-run.json for sanitized prior-run evidence.")

## 5. Upload or reuse the exact validated files

Reusing both file IDs avoids duplicate uploads. When uploading, this cell opens `TRAINING_PATH` and
`VALIDATION_PATH` directly—the same files whose hashes and schemas were validated above.


In [ ]:
if config["existing_job_id"]:
    training_file_id = config["existing_training_file_id"]
    validation_file_id = config["existing_validation_file_id"]
    print("Existing job configured; file upload is not required for replay.")
elif config["existing_training_file_id"]:
    training_file = openai_client.files.retrieve(config["existing_training_file_id"])
    validation_file = openai_client.files.retrieve(config["existing_validation_file_id"])
    training_file_id = training_file.id
    validation_file_id = validation_file.id
    print("Reusing existing project files.")
else:
    with TRAINING_PATH.open("rb") as handle:
        training_file = openai_client.files.create(
            file=(TRAINING_PATH.name, handle),
            purpose="fine-tune",
        )
    with VALIDATION_PATH.open("rb") as handle:
        validation_file = openai_client.files.create(
            file=(VALIDATION_PATH.name, handle),
            purpose="fine-tune",
        )
    training_file_id = training_file.id
    validation_file_id = validation_file.id
    for file_id in (training_file_id, validation_file_id):
        processed = openai_client.files.wait_for_processing(file_id)
        if processed.status != "processed":
            raise RuntimeError(f"File processing failed for {file_id}: {processed.status}")
    print("Uploaded and processed the validated dataset files.")


## 6. Create or reuse the supervised fine-tuning job

Creation is intentionally gated. Set `FOUNDRY_RUN_PAID_JOBS=true` only after reviewing the dataset,
baseline, current pricing, quota, and expected runtime. A configured existing job is always reused.


In [ ]:
if config["existing_job_id"]:
    fine_tune_job = openai_client.fine_tuning.jobs.retrieve(config["existing_job_id"])
    print("Reusing the configured fine-tuning job.")
else:
    if not config["allow_paid_job_creation"]:
        raise RuntimeError(
            "Paid job creation is blocked. Set FOUNDRY_RUN_PAID_JOBS=true only when ready."
        )
    fine_tune_job = openai_client.fine_tuning.jobs.create(
        model=config["sft_base_model"],
        training_file=training_file_id,
        validation_file=validation_file_id,
        method={
            "type": "supervised",
            "supervised": {
                "hyperparameters": {
                    "n_epochs": 1,
                    "batch_size": 1,
                    "learning_rate_multiplier": 1.0,
                }
            },
        },
        suffix="news-summarization-sft",
        extra_body={"trainingType": "GlobalStandard"},
    )
    print("Created a supervised fine-tuning job.")

print(f"Job state: {fine_tune_job.status}")
print(f"Training model: {fine_tune_job.model}")


## 7. Monitor to a terminal state

Polling stops on `succeeded`, `failed`, or `cancelled`. Failures remain failures: the notebook does
not silently retry invalid data, authentication, quota, or safety errors as if they succeeded.


In [ ]:
terminal_states = {"succeeded", "failed", "cancelled"}
seen_event_ids = set()

while True:
    fine_tune_job = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    events = openai_client.fine_tuning.jobs.list_events(fine_tune_job.id, limit=100)
    event_items = getattr(events, "data", events)
    for event in reversed(list(event_items)):
        if event.id not in seen_event_ids:
            seen_event_ids.add(event.id)
            print(f"{event.created_at}: {event.message}")
    print(f"Job state: {fine_tune_job.status}")
    if fine_tune_job.status in terminal_states:
        break
    time.sleep(60)

if fine_tune_job.status != "succeeded":
    raise RuntimeError(
        f"Fine-tuning job ended as {fine_tune_job.status}: {getattr(fine_tune_job, 'error', None)}"
    )
fine_tuned_model_id = fine_tune_job.fine_tuned_model
if not fine_tuned_model_id:
    raise RuntimeError("Succeeded job did not expose a fine-tuned model ID")
print("Fine-tuning succeeded.")


## 8. Resolve or create a meaningful deployment, then invoke it

Reuse `FOUNDRY_FINE_TUNED_DEPLOYMENT` by default. Optional ARM deployment creation requires explicit
configuration and uses a demo-derived name. Resource and deployment identifiers are written only to
ignored runtime output.


In [ ]:
deployment_created_by_notebook = False
management_client = None

if config["fine_tuned_deployment_name"]:
    deployment_name = config["fine_tuned_deployment_name"]
elif config["create_deployment"]:
    from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
    from azure.mgmt.cognitiveservices.models import (
        Deployment,
        DeploymentModel,
        DeploymentProperties,
        Sku,
    )

    required_management = {
        "FOUNDRY_SUBSCRIPTION_ID": config["subscription_id"],
        "FOUNDRY_RESOURCE_GROUP": config["resource_group"],
        "FOUNDRY_DEPLOYMENT_SKU": config["deployment_sku"],
        "FOUNDRY_DEPLOYMENT_CAPACITY": config["deployment_capacity"],
    }
    missing = [name for name, value in required_management.items() if not value]
    if missing:
        raise RuntimeError(f"Deployment creation requires: {missing}")

    project_host = urlparse(config["project_endpoint"]).hostname
    if not project_host or not project_host.endswith(".services.ai.azure.com"):
        raise ValueError("Could not derive the Foundry account name from the project endpoint")
    account_name = project_host.removesuffix(".services.ai.azure.com")
    deployment_name = f"{OPERATION_NAME}-ft-{fine_tune_job.id[-8:]}"
    deployment_options = (fine_tune_job.metadata or {}).get("deployment_options", [])
    deployment_format = deployment_options[0].get("format", "OpenAI") if deployment_options else "OpenAI"

    management_client = CognitiveServicesManagementClient(
        credential=credential,
        subscription_id=config["subscription_id"],
    )
    deployment = Deployment(
        properties=DeploymentProperties(
            model=DeploymentModel(
                format=deployment_format,
                name=fine_tuned_model_id,
                version="1",
            )
        ),
        sku=Sku(
            name=config["deployment_sku"],
            capacity=int(config["deployment_capacity"]),
        ),
    )
    poller = management_client.deployments.begin_create_or_update(
        resource_group_name=config["resource_group"],
        account_name=account_name,
        deployment_name=deployment_name,
        deployment=deployment,
    )
    poller.result()
    deployment_created_by_notebook = True
else:
    raise RuntimeError(
        "Set FOUNDRY_FINE_TUNED_DEPLOYMENT, or explicitly enable FOUNDRY_CREATE_DEPLOYMENT."
    )

resolved_deployment = project_client.deployments.get(deployment_name)
if resolved_deployment.model_name != fine_tuned_model_id:
    raise RuntimeError(
        f"Deployment model mismatch: {resolved_deployment.model_name} != {fine_tuned_model_id}"
    )
print("Fine-tuned deployment resolved and model identity verified.")


## 9. Evaluate with the same contract

The fine-tuned deployment is scored on the exact same examples and metrics as the baseline. For
summarization, overlap metrics are diagnostic rather than sufficient: inspect examples and require
human factuality review before promotion.

Live inference in the next cell runs only when `FOUNDRY_RUN_LIVE_EVALUATION=true`; otherwise it sends no service request and points to checked-in representative evidence.

In [ ]:
fine_tuned_results = None
if config["run_live_evaluation"]:
    fine_tuned_results = evaluate_deployment(
        deployment_name,
        evaluation_examples,
        "fine_tuned",
    )
    print(json.dumps(fine_tuned_results["aggregate"], indent=2))
else:
    print("Live fine-tuned evaluation skipped; no inference requests were sent.")

## 10. Gains summary and decision

The ignored runtime summary captures hashes, current-run identifiers, metrics, deltas, and the
decision. Do not promote a model merely because training succeeded.


In [ ]:
if baseline_results is not None and fine_tuned_results is not None:
    metric_deltas = {
        metric: fine_tuned_results["aggregate"][metric] - baseline_results["aggregate"][metric]
        for metric in baseline_results["aggregate"]
    }
    decision = (
        "promote for human factuality review"
        if metric_deltas["rouge_l_f1"] > 0 and metric_deltas["domain_coverage"] >= 0
        else "do not promote"
    )
else:
    metric_deltas = None
    decision = "live evaluation not run; no promotion decision"
run_summary = {
    "operation": OPERATION_NAME,
    "dataset_hashes": dataset_report["hashes"],
    "job_id": fine_tune_job.id,
    "fine_tuned_model": fine_tuned_model_id,
    "deployment_name": deployment_name,
    "baseline": baseline_results,
    "fine_tuned": fine_tuned_results,
    "deltas": metric_deltas,
    "decision": decision,
    "human_review_required": True,
}
summary_path = OUTPUT_DIR / "run-summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({"decision": decision, "deltas": metric_deltas}, indent=2))
print(f"Saved current-run details to {summary_path}")

## 11. Key lessons and next curriculum step

- Validate and hash the exact files that will be uploaded.
- Establish the evaluation contract before creating a paid job.
- Reuse files, jobs, and deployments deliberately; never confuse successful training with quality.
- Keep service identifiers in ignored runtime output, not committed teaching material.
- This source demo has no independent final holdout. Add one for a real project without changing the
  preserved curriculum datasets.

**Next:** [Domain SFT: Scientific Summarization](../../02-scientific-summarization/README.md)


## 12. Cleanup

By default the notebook retains the deployment for deliberate reuse. Set
`FOUNDRY_DELETE_CREATED_DEPLOYMENT=true` only when this notebook created the deployment and you want to remove
its hosting cost. Project files and fine-tuning jobs remain as service records.


In [ ]:
if deployment_created_by_notebook and config["delete_created_deployment"]:
    delete_poller = management_client.deployments.begin_delete(
        resource_group_name=config["resource_group"],
        account_name=account_name,
        deployment_name=deployment_name,
    )
    delete_poller.result()
    print("Deleted the deployment created by this notebook.")
else:
    print("Deployment retained for reuse.")

if management_client is not None:
    management_client.close()
openai_client.close()
project_client.close()
credential.close()
print("Clients closed.")
